In [3]:
#We will use TensorBoard to track experiments and 
#compare models.

In [4]:
import torch 
import torchvision

In [5]:
import os
import zipfile

from pathlib import Path

import requests

def download_data(source: str, 
                  destination: str,
                  remove_source: bool = True) -> Path:
    """Downloads a zipped dataset from source and unzips to destination.

    Args:
        source (str): A link to a zipped file containing data.
        destination (str): A target directory to unzip data to.
        remove_source (bool): Whether to remove the source after downloading and extracting.
    
    Returns:
        pathlib.Path to downloaded data.
    
    Example usage:
        download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                      destination="pizza_steak_sushi")
    """
    # Setup path to data folder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder doesn't exist, download it and prepare it... 
    if image_path.is_dir():
        print(f"[INFO] {image_path} directory exists, skipping download.")
    else:
        print(f"[INFO] Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)
        
        # Download pizza, steak, sushi data
        target_file = Path(source).name
        with open(data_path / target_file, "wb") as f:
            request = requests.get(source)
            print(f"[INFO] Downloading {target_file} from {source}...")
            f.write(request.content)

        # Unzip pizza, steak, sushi data
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"[INFO] Unzipping {target_file} data...") 
            zip_ref.extractall(image_path)

        # Remove .zip file
        if remove_source:
            os.remove(data_path / target_file)
    
    return image_path

In [6]:
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd 
import seaborn as sns


from torch import nn 
from torchvision import transforms
from torchinfo import summary 

from going_modular import engine ,data_setup 






/Users/swapneelpremchand/PyTorch/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
device = 'mps' if torch.mps.is_available() else 'cpu'

In [8]:
# Set seeds
def set_seeds(seed: int=42):
    """Sets random sets for torch operations.

    Args:
        seed (int, optional): Random seed to set. Defaults to 42.
    """
    # Set the seed for general torch operations
    torch.manual_seed(seed)
    # Set the seed for CUDA torch operations (ones that happen on the GPU)
    torch.mps.manual_seed(seed)

In [9]:
from pathlib import Path 
import os 


image_path = Path("data/pizza_steak_sushi")

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])




In [10]:
train_dir = image_path / "train"

test_dir = image_path / "test"

manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=manual_transforms, # use manually created transforms
    batch_size=32
)

In [11]:

train_dir = image_path / "train"
test_dir = image_path / "test"

# Setup pretrained weights 
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT

# Get transforms from weights 
automatic_transforms = weights.transforms() 
print(f"Automatically created transforms: {automatic_transforms}")

# Create data loaders
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=automatic_transforms, 
    batch_size=32
)

train_dataloader, test_dataloader, class_names

Automatically created transforms: ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)


(<torch.utils.data.dataloader.DataLoader at 0x1057f44c0>,
 ['pizza', 'steak', 'sushi'])

In [12]:


# Download the pretrained weights for EfficientNet_B0
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT 


model = torchvision.models.efficientnet_b0(weights=weights).to(device)


# model

In [13]:
for params in model.features.parameters():
    params.requires_grad = False

In [14]:
model.classifier = torch.nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, 
              out_features=len(class_names),
              bias=True).to(device))

In [15]:
#Uncomment to view , hurts my eyes to see this too many times 

#summary(model , input_size= (32 , 3 , 224 ,224))

In [16]:
# Define loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [17]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter()

In [18]:
# Train model
# Note: Not using engine.train() since the original script isn't updated to use writer
set_seeds()
results = engine.train(model=model,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                epochs=5,
                device=device)

  0%|          | 0/5 [00:00<?, ?it/s]/Users/swapneelpremchand/PyTorch/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/swapneelpremchand/PyTorch/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
 20%|██        | 1/5 [00:02<00:09,  2.37s/it]

Epoch: 1 | train_loss: 1.0676 | train_acc: 0.4258 | test_loss: 0.8680 | test_acc: 0.7841


 40%|████      | 2/5 [00:03<00:05,  1.91s/it]

Epoch: 2 | train_loss: 0.8540 | train_acc: 0.8125 | test_loss: 0.7974 | test_acc: 0.7131


 60%|██████    | 3/5 [00:05<00:03,  1.76s/it]

Epoch: 3 | train_loss: 0.7540 | train_acc: 0.8359 | test_loss: 0.7395 | test_acc: 0.7633


 80%|████████  | 4/5 [00:07<00:01,  1.69s/it]

Epoch: 4 | train_loss: 0.7111 | train_acc: 0.7734 | test_loss: 0.6414 | test_acc: 0.8040


100%|██████████| 5/5 [00:08<00:00,  1.74s/it]

Epoch: 5 | train_loss: 0.6065 | train_acc: 0.8047 | test_loss: 0.6320 | test_acc: 0.8153


In [19]:
results

{'train_loss': [1.0675842389464378,
  0.8539511561393738,
  0.7539781630039215,
  0.7111194431781769,
  0.6065137386322021],
 'train_acc': [0.42578125, 0.8125, 0.8359375, 0.7734375, 0.8046875],
 'test_loss': [0.867978592713674,
  0.7973713874816895,
  0.7395461002985636,
  0.6413623889287313,
  0.6319698492685953],
 'test_acc': [0.7840909090909092,
  0.7130681818181818,
  0.7632575757575758,
  0.8039772727272728,
  0.8153409090909092]}

In [20]:

%load_ext tensorboard
%tensorboard --logdir runs

Reusing TensorBoard on port 6006 (pid 54430), started 9 days, 9:11:04 ago. (Use '!kill 54430' to kill it.)

In [21]:
from torch.utils.tensorboard import SummaryWriter

def create_writer(experiment_name: str, 
                  model_name: str, 
                  extra: str = None) -> SummaryWriter:
    """
    Creates a TensorBoard SummaryWriter with a timestamped directory.
    
    Args:
        experiment_name: Name of the experiment
        model_name: Name of the model being tested
        extra: Optional extra information to add to the log directory name
        
    Returns:
        SummaryWriter: A TensorBoard SummaryWriter object
    """
    from datetime import datetime
    import os

    timestamp = datetime.now().strftime("%Y-%m-%d") # returns current date in YYYY-MM-DD format

    if extra:
        # Create log directory path
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name, extra)
    else:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name)
        
    print(f"[INFO] Created SummaryWriter, saving to: {log_dir}...")
    return SummaryWriter(log_dir=log_dir)
    
    
    


In [22]:
example_writer = create_writer(experiment_name="data_10_percent",
                               model_name="effnetb0",
                               extra="5_epochs")

[INFO] Created SummaryWriter, saving to: runs/2026-03-22/data_10_percent/effnetb0/5_epochs...


In [24]:
# Download 10 percent and 20 percent training data (if necessary)
data_10_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                                     destination="pizza_steak_sushi")

data_20_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
                                     destination="pizza_steak_sushi_20_percent")

[INFO] data/pizza_steak_sushi directory exists, skipping download.
[INFO] data/pizza_steak_sushi_20_percent directory exists, skipping download.


In [25]:
# Setup training directory paths
train_dir_10_percent = data_10_percent_path / "train"
train_dir_20_percent = data_20_percent_path / "train"

# Setup testing directory paths 
test_dir = data_10_percent_path / "test"

# Check the directories
print(f"Training directory 10%: {train_dir_10_percent}")
print(f"Training directory 20%: {train_dir_20_percent}")
print(f"Testing directory: {test_dir}")

Training directory 10%: data/pizza_steak_sushi/train
Training directory 20%: data/pizza_steak_sushi_20_percent/train
Testing directory: data/pizza_steak_sushi/test
